## Portfolio 2

In deze portfolio gaan we prompt engineering toepassen om de data van de HBO Kennisbank te webscrapen.

Daarna gaan we met deze data titelgeneratie toepassen. Dit gaan we verder finetunen, sampelen, RLAIF en evalueren. 

### Prompt 1

### Antwoord 1

### Resultaat verwerken

In [ ]:
# stap 2
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup

url = "https://hbo-kennisbank.nl/searchresult?q="
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

titles = soup.find_all("h2")  # voorbeeld, moet je inspecteren

for t in titles:
    print(t.get_text(strip=True))

# stap 3
base_url = "https://hbo-kennisbank.nl/searchresult?page={}"

for page in range(1, 50):
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # parse titles

# stap 4
links = []

for item in soup.select("a.result-link"):
    links.append(item["href"])

# stap 5
def scrape_detail(url):
    r = requests.get(url, headers=headers)
    s = BeautifulSoup(r.text, "html.parser")

    title = s.find("h1").get_text(strip=True)
    abstract = s.find("div", class_="abstract")

    return {
        "title": title,
        "abstract": abstract.get_text(strip=True) if abstract else None
    }

# stap 6

data = []

for link in links:
    try:
        item = scrape_detail(link)
        data.append(item)
    except Exception as e:
        print("Error:", e)

# stap 7
time.sleep(1)

# stap 8

df = pd.DataFrame(data)
df.to_csv("hbo_kennisbank.csv", index=False)

### Prompt 2

### Antwoord 2

### Resultaat verwerken

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = "https://hbo-kennisbank.nl/searchresult?page={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
}

# ---------------------------
# Helper: veilige request
# ---------------------------
def get_soup(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        print(f"[ERROR] {url} -> {e}")
        return None


# ---------------------------
# Stap 1: Verzamel links
# ---------------------------
def extract_links(soup):
    links = []

    # meerdere fallback selectors
    selectors = [
        "a.search-result__title",
        "a.result-item__title",
        "h2 a",
        "a[href*='/details']"
    ]

    for selector in selectors:
        elements = soup.select(selector)
        if elements:
            for el in elements:
                href = el.get("href")
                if href:
                    full_url = urljoin(BASE_URL, href)
                    links.append(full_url)
            break

    return list(set(links))  # dedupliceren


# ---------------------------
# Stap 2: Detailpagina scrapen
# ---------------------------
def scrape_detail(url):
    soup = get_soup(url)
    if not soup:
        return None

    def safe_text(element):
        return element.get_text(strip=True) if element else None

    # Titel
    title = None
    for sel in ["h1", ".publication-title"]:
        el = soup.select_one(sel)
        if el:
            title = safe_text(el)
            break

    # Abstract
    abstract = None
    for sel in [".abstract", ".description", "#abstract"]:
        el = soup.select_one(sel)
        if el:
            abstract = safe_text(el)
            break

    # Auteurs
    authors = None
    for sel in [".authors", ".creator", ".author"]:
        el = soup.select_one(sel)
        if el:
            authors = safe_text(el)
            break

    # Jaar
    year = None
    for sel in [".year", ".date", ".publication-date"]:
        el = soup.select_one(sel)
        if el:
            year = safe_text(el)
            break

    return {
        "title": title,
        "abstract": abstract,
        "authors": authors,
        "year": year,
        "url": url
    }


# ---------------------------
# Stap 3: Hoofd scraper
# ---------------------------
def scrape_knowledgebank(max_pages=10, delay_range=(1, 2)):
    all_links = []

    print("🔎 Links verzamelen...")

    for page in tqdm(range(1, max_pages + 1)):
        url = SEARCH_URL.format(page)
        soup = get_soup(url)

        if not soup:
            continue

        links = extract_links(soup)
        all_links.extend(links)

        time.sleep(random.uniform(*delay_range))

    all_links = list(set(all_links))
    print(f"✅ {len(all_links)} unieke links gevonden")

    print("📄 Detailpagina's scrapen...")
    data = []

    for link in tqdm(all_links):
        item = scrape_detail(link)

        if item and item["title"]:
            data.append(item)

        time.sleep(random.uniform(*delay_range))

    return pd.DataFrame(data)


# ---------------------------
# RUN
# ---------------------------
if __name__ == "__main__":
    df = scrape_knowledgebank(max_pages=20)

    print("\n📊 Resultaat:")
    print(df.head())

    df.to_csv("hbo_kennisbank.csv", index=False)
    print("\n💾 Opgeslagen als hbo_kennisbank.csv")

🔎 Links verzamelen...


100%|██████████| 20/20 [00:37<00:00,  1.89s/it]


✅ 10 unieke links gevonden
📄 Detailpagina's scrapen...


100%|██████████| 10/10 [00:18<00:00,  1.85s/it]



📊 Resultaat:
                                               title abstract authors  year  \
0  Boksen voor mentaal welzijn: Challenge, Change...     None    None  None   
1  Van Lineaire Ontwerpketens naar Circulaire Ont...     None    None  None   
2                                 Koelen in Woningen     None    None  None   
3  Talentontwikkeling en kansengelijkheid in het ...     None    None  None   
4  Understanding shelters as children’s places : ...     None    None  None   

                                                 url  
0  https://hbo-kennisbank.nl/details/sharekit_inh...  
1  https://hbo-kennisbank.nl/details/sharekit_win...  
2  https://hbo-kennisbank.nl/details/hanze_pure_r...  
3  https://hbo-kennisbank.nl/details/hanze_pure_r...  
4  https://hbo-kennisbank.nl/details/sharekit_win...  


OSError: [Errno 30] Read-only file system: 'hbo_kennisbank.csv'